# Usage Metadata Callback Reference

Developer-facing statements defined in `langchain_core.callbacks.usage`.

# `UsageMetadataCallbackHandler: BaseCallbackHandler`

Callback handler that tracks `AIMessage.usage_metadata` across completed chat-model calls.

Added in `langchain-core` 0.3.49.

## Fields

```python
usage_metadata: dict[str, UsageMetadata] # Aggregated usage metadata keyed by model name
```

## Constructor

```python
UsageMetadataCallbackHandler() -> None # Initialize an empty usage-metadata tracker
```

## Methods

### `on_llm_end`

Collects usage metadata from the first chat generation in a completed LLM result.

Usage is recorded only when the generation contains an `AIMessage` with both usage metadata and a model name. Repeated results for the same model are combined.

```python
on_llm_end(
    response: LLMResult, # Completed LLM result whose usage metadata is collected
    **kwargs: Any, # Additional callback metadata
) -> None
```

---

# `get_usage_metadata_callback`

Creates a context manager that tracks usage metadata across chat-model calls made within its context.

Added in `langchain-core` 0.3.49.

```python
@contextmanager
get_usage_metadata_callback(
    name: str = "usage_metadata_callback", # Name of the context variable
) -> Generator[UsageMetadataCallbackHandler, None, None] # Context-managed usage callback
```

The yielded `UsageMetadataCallbackHandler` is automatically registered as an inheritable callback for the duration of the context.

In [ ]:
# Install once in Jupyter Notebook if required.
# !pip install langchain-core

from langchain_core.callbacks.usage import UsageMetadataCallbackHandler
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, LLMResult


def create_llm_result(
    model_name: str,
    input_tokens: int,
    output_tokens: int,
    response_text: str,
) -> LLMResult:
    """Create a simulated LangChain LLM result containing token usage."""

    message = AIMessage(
        content=response_text,
        response_metadata={
            "model_name": model_name
        },
        usage_metadata={
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
        },
    )

    generation = ChatGeneration(message=message)

    return LLMResult(
        generations=[[generation]]
    )


# Create the usage-tracking callback.
usage_callback = UsageMetadataCallbackHandler()


# Simulated first call to demo-model-a.
result_1 = create_llm_result(
    model_name="demo-model-a",
    input_tokens=20,
    output_tokens=10,
    response_text="Python is a high-level programming language.",
)

usage_callback.on_llm_end(result_1)


# Simulated second call to the same model.
result_2 = create_llm_result(
    model_name="demo-model-a",
    input_tokens=15,
    output_tokens=8,
    response_text="LangChain helps build LLM-powered applications.",
)

usage_callback.on_llm_end(result_2)


# Simulated call to a different model.
result_3 = create_llm_result(
    model_name="demo-model-b",
    input_tokens=30,
    output_tokens=12,
    response_text="Callbacks can monitor model execution.",
)

usage_callback.on_llm_end(result_3)


# Display usage grouped by model.
print("Usage metadata by model:\n")

for model_name, usage in usage_callback.usage_metadata.items():
    print(f"Model: {model_name}")
    print(f"Input tokens:  {usage['input_tokens']}")
    print(f"Output tokens: {usage['output_tokens']}")
    print(f"Total tokens:  {usage['total_tokens']}")
    print("-" * 35)


# Display the complete callback state.
print("\nComplete usage dictionary:")
print(usage_callback.usage_metadata)